[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/07_compare_runs.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# Compare runs

**Rung 3 to 4 of 5 (next: rung 4, [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) or
[09, Scale](09_scale.ipynb)). You are here: two traces of the same model, read side by side.**
Trigger: you ran it twice.

A `Bundle` is a dict of traces that TorchLens has aligned op by op. Once two runs sit in one
bundle you can ask where they differ, how much, and in which order the difference spreads. The
sweep from 05 already handed you one; this notebook reads it. At the end the bundle also explains
how each member was made and goes to disk as one object.

> **You'll want to know:** [`intervene=`](05_change_and_rerun.ipynb) (an edit during the forward
> pass), [labels](02_address_and_read.ipynb) (`relu_1_2` names a site),
> [`tl.sweep`](05_change_and_rerun.ipynb) (one site, several values, one bundle back).

In [2]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [3]:
from learn_models import tiny_mlp  # tiny seeded CPU models; nothing downloads

model, x = tiny_mlp()  # two linear layers with a ReLU between them, and a (4, 8) input batch

Two traces of the same model, one clean and one with its ReLUs zeroed, ranked by how far each
site moved, largest first. Sites upstream of the edit do not move at all. That is the whole
notebook in one cell.


In [4]:
clean = tl.trace(model, x)
ablated = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.zero_ablate()))
pair = tl.Bundle({"clean": clean, "ablated": ablated}, baseline="clean")
for site, distance in pair.rank_changes(k=3):
    print(f"{site:>12}: {distance:.3f}")

  linear_2_3: 1.171
    output_1: 1.171
    relu_1_2: 1.000


## 1. A bundle is a dict of traces

Index it by member name and you get the trace back. The `baseline=` member is the one every
other member is measured against. `comparability` says how two members' models relate before
any number is computed; here they share one model object. Whether their inputs match is a
separate check, made when a read needs it (section 4).


In [5]:
print(
    type(pair["clean"]).__name__,
    "|",
    len(pair),
    "members:",
    pair.names,
    "| baseline:",
    pair.baseline_name,
)
print(pair.comparability("clean", "ablated"))

Trace | 2 members: ['clean', 'ablated'] | baseline: clean
Relationship.SAME_OBJECT


## 2. Site by site

`compare(a, b)` walks every site the two members share and returns `(site, distance)` rows in
dataflow order; the default metric is cosine distance. `at(site)` turns the axis around: one site,
every member. `metric=` picks the distance:
`compare(a, b, metric="relative_l2")` is the L2 distance from `a`, relative to `a`'s norm, site
by site, the output included.


In [6]:
for site, distance in pair.compare("clean", "ablated")[:4]:
    print(f"{site:>12}: {abs(distance):.3f}")

     input_1: 0.000
  linear_1_1: 0.000
    relu_1_2: 1.000
  linear_2_3: 1.171


In [7]:
print("rows and columns:", pair.names)
pair.at("relu_1_2").compare()  # the member-by-member matrix at one site

rows and columns: ['clean', 'ablated']


tensor([[0., 1.],
        [1., 0.]])

In [8]:
deltas = dict(pair.compare("clean", "ablated", metric="relative_l2"))  # {site: relative L2}
for site in ["linear_1_1", "relu_1_2", "linear_2_3", "output_1"]:
    print(f"{site:>12}: {deltas[site]:.3f}")

  linear_1_1: 0.000
    relu_1_2: 1.000
  linear_2_3: 1.056
    output_1: 1.056


## 3. Many members from one sweep

`tl.sweep` built a bundle in 05. The site is a capture-time selector, the same `tl.func("relu")`
that chose what to save and what to edit; members are named by position unless you pass `names=`,
and `include_baseline=True` makes the untouched run the baseline. The same reads work
unchanged: `rank_changes` now scores each site averaged over every member, and `compare`
against the baseline stays per member.

In [9]:
grid = tl.sweep(model, x, tl.func("relu"), values=[0.0, 0.5, 1.0, 2.0], include_baseline=True)
print("members:", grid.names, "| baseline:", grid.baseline_name)
for site, distance in grid.rank_changes(k=3):
    print(f"{site:>12}: {distance:.3f}")
output_moves = {
    member: dict(grid.compare(grid.baseline_name, member, metric="relative_l2"))["output_1"]
    for member in grid.names[1:]
}
print(
    "output move per member:", {member: round(value, 3) for member, value in output_moves.items()}
)

members: ['baseline', 'sweep_0', 'sweep_1', 'sweep_2', 'sweep_3'] | baseline: baseline
    relu_1_2: 0.577
  linear_2_3: 0.411
    output_1: 0.411
output move per member: {'sweep_0': 1.056, 'sweep_1': 0.738, 'sweep_2': 2.075, 'sweep_3': 4.984}


## 4. Two inputs, same model

A bundle does not need an edit. Two runs on different inputs align the same way, and
`comparability` still reports the model relationship, the same object. `compare` lines the
sites up by graph shape when the inputs differ, so each distance is the input's effect at that
site; it still refuses with a typed error when data-dependent control flow took different
branches. `map` applies any per-member read across the bundle and is the cross-input tool.

In [10]:
other = tl.trace(model, torch.randn(4, 8, generator=torch.Generator().manual_seed(1)))
inputs = tl.Bundle({"x": clean, "x2": other}, baseline="x")
print(inputs.comparability("x", "x2"))
across = dict(inputs.compare("x", "x2", metric="relative_l2"))
print("output move across inputs:", round(across["output_1"], 3))
print(
    "output norm per member:",
    inputs.map(lambda t: round(float(t.reconstruct_output().detach().norm()), 3)),
)

Relationship.SAME_OBJECT
output move across inputs: 0.696
output norm per member: {'x': 1.03, 'x2': 1.052}


## 5. Provenance, and the bundle on disk

Every member remembers how it was made. `why(member)` answers "this member is the baseline plus
which edits", `provenance()` is the same fact as one row per member, and `save` writes the whole
bundle as one directory that `tl.load` brings back as a `Bundle`.

In [11]:
print(grid.why(grid.names[1]).describe())

'sweep_0' = 'baseline' + [helper:sweep_replace:(0.0,):() @ selector:tl.func('relu') (capture/trace, fired)] (lineage exact, basis=container_operation, fidelity=declared)


In [12]:
for row in grid.provenance():
    print(row["member"], "|", row["lineage_status"])

baseline | baseline
sweep_0 | exact
sweep_1 | exact
sweep_2 | exact
sweep_3 | exact


In [13]:
with tempfile.TemporaryDirectory() as tmp:
    grid.save(Path(tmp) / "sweep.tlspec", overwrite=True)
    loaded = tl.load(Path(tmp) / "sweep.tlspec")
    print(type(loaded).__name__, loaded.names, "| baseline:", loaded.baseline_name)

Bundle ['baseline', 'sweep_0', 'sweep_1', 'sweep_2', 'sweep_3'] | baseline: baseline


## 6. That was rung 4's first half

`tl.Bundle({...}, baseline=...)` aligns traces; `rank_changes`, `compare`, `at(site).compare()`
and the delta family read them; `tl.sweep` makes many members at once; `why` and `provenance`
remember how; `save` and `tl.load` carry the whole experiment.

**One thing about networks:** an intervention's footprint is its downstream cone: nothing
upstream moves, everything downstream does. `compare` walks that cone in dataflow order and
`rank_changes` sorts it by distance; a bundle is how you see the cone as numbers rather than as a
diagram.

**Go deeper:** next is [08, Semantic shortcuts](08_semantic_shortcuts.ipynb) for attention
models, or [09, Scale](09_scale.ipynb) for thousands of batches. Workflows:
[ablations_grid](../../examples/50min/ablations_grid.ipynb) and
[paired_prompt_patching](../../examples/50min/paired_prompt_patching.ipynb). Reference:
[Experiments remember](../../docs/reference/experiments.md) (the Bundle is the record) and the
glossary's "Bundle" entries in [glossary.md](../../docs/reference/glossary.md).